In [1]:
!pip install pymongo pandas


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
from pymongo import MongoClient
from datetime import datetime

In [3]:
MONGO_URI = "mongodb://localhost:27017"

client = MongoClient(MONGO_URI)

db = client["amazon_reviews_db"]

print("Connected to MongoDB Atlas!")

Connected to MongoDB Atlas!


## Load Sample Dataset

In [4]:
df = pd.read_json("../data/processed/50_000_sample_dataset_processed.json", lines=True)

print(df.shape)
df.head()

(67377, 12)


,overall,vote,verified,reviewTime,reviewerID,asin,style,reviewerName,reviewText,summary,unixReviewTime,image
0,5,0,True,"10 10, 2014",A3VUZL5BTT44GI,B00004T8R2,{'Style:': ' On Ear'},chertheatr,"Affordable, light, not super easy to break. Se...",Preschool perfect,1412899200,None
1,5,0,True,"03 6, 2013",A99D8N0YVVDOC,B00001WRSJ,{'Product Packaging:': ' Standard Packaging'},Bill,"Great sound quality, great value. Highly recom...",Great Headphones,1362528000,None
2,4,0,False,"08 13, 2006",A29HQXXCBUHJL8,B00004Z0BO,{'Product Packaging:': ' Standard Packaging'},Jay,"Well, I am not a bass friend but I would like ...",A good earphone without enough bass,1155427200,None
3,1,0,True,"08 6, 2013",A1MG4LGGWCFOHM,B00004ZCJI,"{'Size:': ' 82mm', 'Package Type:': ' Standard...",Ronald Olivier,I never recieved it but was charged! What are ...,Why I should not buy from Amazon again!!!!!!!!...,1375747200,None
4,5,0,True,"12 7, 2015",A2NNXYE8E3XK3M,B00004Z5UM,{'Length:': ' Without CE'},Arthur B.,Works fine.,No problems.,1449446400,None


## Create Collections

In [5]:
users_col = db["users"]
products_col = db["products"]
reviews_col = db["reviews"]

print("Collections ready")

Collections ready


## Build Users Collection (1 → Many)

In [6]:
users = {}

for _, row in df.iterrows():
    user_id = row["reviewerID"]
    
    if user_id not in users:
        users[user_id] = {
            "_id": user_id,
            "name": row.get("reviewerName", "Unknown"),
            "profile": {
                "total_reviews": 0,
                "avg_rating": 0
            }
        }

    # Update profile stats
    users[user_id]["profile"]["total_reviews"] += 1

# Calculate average rating
for _, row in df.iterrows():
    user_id = row["reviewerID"]
    users[user_id]["profile"]["avg_rating"] += row["overall"]

for user in users.values():
    total = user["profile"]["total_reviews"]
    user["profile"]["avg_rating"] /= total

# Insert into MongoDB
users_col.insert_many(list(users.values()))

print("Users inserted:", users_col.count_documents({}))

Users inserted: 62846


## Build Products Collection (1 → Many)

In [7]:
products = {}

for _, row in df.iterrows():
    asin = row["asin"]
    
    if asin not in products:
        products[asin] = {
            "_id": asin,
            "style": row.get("style", {})
        }

products_col.insert_many(list(products.values()))

print("Products inserted:", products_col.count_documents({}))

Products inserted: 34291


## Build Reviews Collection (Many ↔ Many)

In [8]:
reviews = []

for _, row in df.iterrows():
    review_doc = {
        "reviewerID": row["reviewerID"],  
        "asin": row["asin"],              
        "overall": float(row["overall"]),
        "vote": int(row.get("vote", 0)),
        "verified": bool(row["verified"]),
        "reviewText": row.get("reviewText", ""),
        "summary": row.get("summary", ""),
        "reviewTime": row["reviewTime"],
        "unixReviewTime": int(row["unixReviewTime"])
    }
    
    reviews.append(review_doc)

batch_size = 1000
for i in range(0, len(reviews), batch_size):
    reviews_col.insert_many(reviews[i:i+batch_size])

print("Reviews inserted:", reviews_col.count_documents({}))

Reviews inserted: 67377
